# 🥈 Camada Silver: Transformação e Padronização

Nesta etapa, processamos os dados brutos da camada Bronze para criar tabelas confiáveis e otimizadas.

**Estratégias aplicadas em todas as pipelines:**
1.  **Schema Enforcement:** Conversão de tipos (`String` -> `Int`, `Date`, `Float`) para garantir consistência numérica.
2.  **Deduplicação:** Remoção de registros duplicados baseados em chaves únicas (IDs).
3.  **Metadados:** Adição de colunas de auditoria (`ingestion_date` e `source_file`).
4.  **Formato Delta:** Salvamento em formato Delta Lake para permitir *ACID Transactions* e performance.


In [0]:
from pyspark.sql.functions import col, explode, current_timestamp, input_file_name 

# --- CONFIGURAÇÃO DE CAMINHOS ---
# PASSO 1: CONFIGURAÇÃO DE ACESSO AO AZURE
# Preencha com seus dados do Azure 
storage_account_name = "f1datalakecarol2026"
storage_account_key = "6FQP+MNUBoy3xCDvI7ZBjUH7IoJqbY7lS82W0seE2S2M5gCpVVhIQfWoZ/FTZAmVXs2E6WR7p2PY+AStKv01ig=="
container_name = "bronze"

# Configura o Spark (Mounting)
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

container_bronze = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
container_silver = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"

print("🚀 Iniciando Processamento SILVER...")


🚀 Iniciando Processamento SILVER...


### 🏎️ Pipeline 1: Drivers (Dimensão)
Tratamento da lista de pilotos. Aplainamento (Flattening) da estrutura aninhada e geração da tabela de dimensão.

In [0]:

print("Iniciando Inspeção dos PILOTOS...")
# Analisando a estrutura do arquivo Raw para planejar a transformação
try:
    df_drivers_raw = spark.read.json(f"{container_bronze}/jolpica/drivers_*.json")
    
    print("--- Schema Identificado ---")
    df_drivers_raw.printSchema()
    
    print("--- Amostra de Dados ---")
    display(df_drivers_raw.limit(5))
except Exception as e:
    print(f"Dados ainda não disponíveis: {e}")

Iniciando Inspeção dos PILOTOS...
--- Schema Identificado ---
root
 |-- MRData: struct (nullable = true)
 |    |-- DriverTable: struct (nullable = true)
 |    |    |-- Drivers: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |-- dateOfBirth: string (nullable = true)
 |    |    |    |    |-- driverId: string (nullable = true)
 |    |    |    |    |-- familyName: string (nullable = true)
 |    |    |    |    |-- givenName: string (nullable = true)
 |    |    |    |    |-- nationality: string (nullable = true)
 |    |    |    |    |-- permanentNumber: string (nullable = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |-- season: string (nullable = true)
 |    |-- limit: string (nullable = true)
 |    |-- offset: string (nullable = true)
 |    |-- series: string (nullable = true)
 |    |-- total: string (nullable = true)
 |    |-- url: string (nullable = tr

MRData
"List(List(List(List(ALB, 1996-03-23, albon, Albon, Alexander, Thai, 23, http://en.wikipedia.org/wiki/Alexander_Albon), List(ALO, 1981-07-29, alonso, Alonso, Fernando, Spanish, 14, http://en.wikipedia.org/wiki/Fernando_Alonso), List(ANT, 2006-08-25, antonelli, Antonelli, Andrea Kimi, Italian, 12, https://en.wikipedia.org/wiki/Andrea_Kimi_Antonelli), List(BEA, 2005-05-08, bearman, Bearman, Oliver, British, 87, http://en.wikipedia.org/wiki/Oliver_Bearman), List(BOT, 1989-08-28, bottas, Bottas, Valtteri, Finnish, 77, http://en.wikipedia.org/wiki/Valtteri_Bottas), List(COL, 2003-05-27, colapinto, Colapinto, Franco, Argentine, 43, http://en.wikipedia.org/wiki/Franco_Colapinto), List(DOO, 2003-01-20, doohan, Doohan, Jack, Australian, 7, http://en.wikipedia.org/wiki/Jack_Doohan), List(GAS, 1996-02-07, gasly, Gasly, Pierre, French, 10, http://en.wikipedia.org/wiki/Pierre_Gasly), List(HAM, 1985-01-07, hamilton, Hamilton, Lewis, British, 44, http://en.wikipedia.org/wiki/Lewis_Hamilton), List(HUL, 1987-08-19, hulkenberg, Hülkenberg, Nico, German, 27, http://en.wikipedia.org/wiki/Nico_H%C3%BClkenberg), List(LAW, 2002-02-11, lawson, Lawson, Liam, New Zealander, 30, http://en.wikipedia.org/wiki/Liam_Lawson), List(LEC, 1997-10-16, leclerc, Leclerc, Charles, Monegasque, 16, http://en.wikipedia.org/wiki/Charles_Leclerc), List(MAG, 1992-10-05, kevin_magnussen, Magnussen, Kevin, Danish, 20, http://en.wikipedia.org/wiki/Kevin_Magnussen), List(NOR, 1999-11-13, norris, Norris, Lando, British, 4, http://en.wikipedia.org/wiki/Lando_Norris), List(OCO, 1996-09-17, ocon, Ocon, Esteban, French, 31, http://en.wikipedia.org/wiki/Esteban_Ocon), List(PIA, 2001-04-06, piastri, Piastri, Oscar, Australian, 81, http://en.wikipedia.org/wiki/Oscar_Piastri), List(PER, 1990-01-26, perez, Pérez, Sergio, Mexican, 11, http://en.wikipedia.org/wiki/Sergio_P%C3%A9rez), List(RIC, 1989-07-01, ricciardo, Ricciardo, Daniel, Australian, 3, http://en.wikipedia.org/wiki/Daniel_Ricciardo), List(RUS, 1998-02-15, russell, Russell, George, British, 63, http://en.wikipedia.org/wiki/George_Russell_(racing_driver)), List(SAI, 1994-09-01, sainz, Sainz, Carlos, Spanish, 55, http://en.wikipedia.org/wiki/Carlos_Sainz_Jr.), List(SAR, 2000-12-31, sargeant, Sargeant, Logan, American, 2, http://en.wikipedia.org/wiki/Logan_Sargeant), List(STR, 1998-10-29, stroll, Stroll, Lance, Canadian, 18, http://en.wikipedia.org/wiki/Lance_Stroll), List(TSU, 2000-05-11, tsunoda, Tsunoda, Yuki, Japanese, 22, http://en.wikipedia.org/wiki/Yuki_Tsunoda), List(VER, 1997-09-30, max_verstappen, Verstappen, Max, Dutch, 3, http://en.wikipedia.org/wiki/Max_Verstappen), List(ZHO, 1999-05-30, zhou, Zhou, Guanyu, Chinese, 24, http://en.wikipedia.org/wiki/Zhou_Guanyu)), 2024), 100, 0, f1, 25, https://api.jolpi.ca/ergast/f1/2024/drivers/, )"


In [0]:
print("🚀 Processando Drivers para Silver...")

try:
    if 'df_drivers_raw' not in locals():
        print("⚠️ Leitura raw não encontrada na memória, lendo arquivos agora...")
        df_drivers_raw = spark.read.json(f"{container_bronze}/jolpica/drivers_*.json")

    df_drivers_exploded = df_drivers_raw \
        .select(explode(col("MRData.DriverTable.Drivers")).alias("driver_data"))

    df_drivers_silver = df_drivers_exploded.select(
        col("driver_data.driverId").alias("driver_id"),
        col("driver_data.givenName").alias("driver_name"),
        col("driver_data.familyName").alias("driver_surname"),
        col("driver_data.nationality").alias("driver_nationality"),
        col("driver_data.dateOfBirth").cast("date").alias("driver_dob"),
        col("driver_data.code").alias("driver_code"),
        col("driver_data.permanentNumber").cast("int").alias("driver_number"),
        col("driver_data.url").alias("driver_url"),
        current_timestamp().alias("ingestion_date"),
        input_file_name().alias("source_file")
    )

    # DEDUPLICAÇÃO
    df_drivers_silver = df_drivers_silver.dropDuplicates(['driver_id'])

    df_drivers_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{container_silver}/drivers")
        
    print("✅ Tabela 'drivers' salva e deduplicada!")    
    print("Preview Drivers:")
    display(df_drivers_silver.limit(5))

except Exception as e:
    print(f"❌ Erro em Drivers: {e}")

🚀 Processando Drivers para Silver...
✅ Tabela 'drivers' salva e deduplicada!
Preview Drivers:


driver_id,driver_name,driver_surname,driver_nationality,driver_dob,driver_code,driver_number,driver_url,ingestion_date,source_file
albon,Alexander,Albon,Thai,1996-03-23,ALB,23,http://en.wikipedia.org/wiki/Alexander_Albon,2026-01-03T17:56:49.568895Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/drivers_2024_20260103.json/part-00000-tid-3272188292792119646-aa899d24-a570-4316-bc88-2c1a24a2c482-14-1-c000.json
alonso,Fernando,Alonso,Spanish,1981-07-29,ALO,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2026-01-03T17:56:49.568895Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/drivers_2024_20260103.json/part-00000-tid-3272188292792119646-aa899d24-a570-4316-bc88-2c1a24a2c482-14-1-c000.json
antonelli,Andrea Kimi,Antonelli,Italian,2006-08-25,ANT,12,https://en.wikipedia.org/wiki/Andrea_Kimi_Antonelli,2026-01-03T17:56:49.568895Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/drivers_2024_20260103.json/part-00000-tid-3272188292792119646-aa899d24-a570-4316-bc88-2c1a24a2c482-14-1-c000.json
bearman,Oliver,Bearman,British,2005-05-08,BEA,87,http://en.wikipedia.org/wiki/Oliver_Bearman,2026-01-03T17:56:49.568895Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/drivers_2024_20260103.json/part-00000-tid-3272188292792119646-aa899d24-a570-4316-bc88-2c1a24a2c482-14-1-c000.json
bottas,Valtteri,Bottas,Finnish,1989-08-28,BOT,77,http://en.wikipedia.org/wiki/Valtteri_Bottas,2026-01-03T17:56:49.568895Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/drivers_2024_20260103.json/part-00000-tid-3272188292792119646-aa899d24-a570-4316-bc88-2c1a24a2c482-14-1-c000.json


### 🔧 Pipeline 2: Constructors (Dimensão)
Tratamento das equipes construtoras.

In [0]:
print("🔍 Iniciando Inspeção das EQUIPES...")

try:
    df_constructors_raw = spark.read.json(f"{container_bronze}/jolpica/constructors_*.json")

    print("📜 Schema Constructors:")
    df_constructors_raw.printSchema()

    print("👀 Amostra Raw:")
    display(df_constructors_raw.limit(5))

except Exception as e:
    print(f"⚠️ Erro na leitura de Constructors: {e}")

🔍 Iniciando Inspeção das EQUIPES...
📜 Schema Constructors:
root
 |-- MRData: struct (nullable = true)
 |    |-- ConstructorTable: struct (nullable = true)
 |    |    |-- Constructors: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- constructorId: string (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |    |-- nationality: string (nullable = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |    |-- season: string (nullable = true)
 |    |-- limit: string (nullable = true)
 |    |-- offset: string (nullable = true)
 |    |-- series: string (nullable = true)
 |    |-- total: string (nullable = true)
 |    |-- url: string (nullable = true)
 |    |-- xmlns: string (nullable = true)

👀 Amostra Raw:


MRData
"List(List(List(List(alpine, Alpine F1 Team, French, http://en.wikipedia.org/wiki/Alpine_F1_Team), List(aston_martin, Aston Martin, British, http://en.wikipedia.org/wiki/Aston_Martin_in_Formula_One), List(ferrari, Ferrari, Italian, http://en.wikipedia.org/wiki/Scuderia_Ferrari), List(haas, Haas F1 Team, American, http://en.wikipedia.org/wiki/Haas_F1_Team), List(mclaren, McLaren, British, http://en.wikipedia.org/wiki/McLaren), List(mercedes, Mercedes, German, http://en.wikipedia.org/wiki/Mercedes-Benz_in_Formula_One), List(rb, RB F1 Team, Italian, http://en.wikipedia.org/wiki/RB_Formula_One_Team), List(red_bull, Red Bull, Austrian, http://en.wikipedia.org/wiki/Red_Bull_Racing), List(sauber, Sauber, Swiss, http://en.wikipedia.org/wiki/Sauber_Motorsport), List(williams, Williams, British, http://en.wikipedia.org/wiki/Williams_Grand_Prix_Engineering)), 2024), 100, 0, f1, 10, https://api.jolpi.ca/ergast/f1/2024/constructors/, )"


In [0]:
print("🚀 Processando Constructors para Silver...")

try:
    df_constructors_exploded = df_constructors_raw \
        .select(explode(col("MRData.ConstructorTable.Constructors")).alias("team_data"))

    df_constructors_silver = df_constructors_exploded.select(
        col("team_data.constructorId").alias("team_id"),
        col("team_data.name").alias("team_name"),
        col("team_data.nationality").alias("team_nationality"),
        col("team_data.url").alias("team_url"),
        current_timestamp().alias("ingestion_date"),
        input_file_name().alias("source_file")
    )
    
    df_constructors_silver = df_constructors_silver.dropDuplicates(['team_id'])

    df_constructors_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{container_silver}/constructors")
        
    print("✅ Tabela 'constructors' salva com sucesso!")
    print("Preview Constructors:")
    display(df_constructors_silver.limit(5))

except Exception as e:
    print(f"❌ Erro em Constructors: {e}")

🚀 Processando Constructors para Silver...
✅ Tabela 'constructors' salva com sucesso!
Preview Constructors:


team_id,team_name,team_nationality,team_url,ingestion_date,source_file
alpine,Alpine F1 Team,French,http://en.wikipedia.org/wiki/Alpine_F1_Team,2026-01-03T17:57:21.367701Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/constructors_2024_20260103.json/part-00000-tid-6788732525327190029-1b355935-432e-4faf-a2d1-d59e7b1eb034-19-1-c000.json
aston_martin,Aston Martin,British,http://en.wikipedia.org/wiki/Aston_Martin_in_Formula_One,2026-01-03T17:57:21.367701Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/constructors_2024_20260103.json/part-00000-tid-6788732525327190029-1b355935-432e-4faf-a2d1-d59e7b1eb034-19-1-c000.json
ferrari,Ferrari,Italian,http://en.wikipedia.org/wiki/Scuderia_Ferrari,2026-01-03T17:57:21.367701Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/constructors_2024_20260103.json/part-00000-tid-6788732525327190029-1b355935-432e-4faf-a2d1-d59e7b1eb034-19-1-c000.json
haas,Haas F1 Team,American,http://en.wikipedia.org/wiki/Haas_F1_Team,2026-01-03T17:57:21.367701Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/constructors_2024_20260103.json/part-00000-tid-6788732525327190029-1b355935-432e-4faf-a2d1-d59e7b1eb034-19-1-c000.json
mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren,2026-01-03T17:57:21.367701Z,abfss://bronze@f1datalakecarol2026.dfs.core.windows.net/jolpica/constructors_2024_20260103.json/part-00000-tid-6788732525327190029-1b355935-432e-4faf-a2d1-d59e7b1eb034-19-1-c000.json


### 🏁 Pipeline 3: Results (Fato)
Tabela principal contendo os resultados das corridas.
* **Nota Técnica:** Aplicamos **Particionamento por Temporada (`season`)** nesta tabela para otimizar a performance de leitura em grandes volumes de dados.


In [0]:
print("🔍 Iniciando Inspeção dos Resultados...")

try:
    path_results = f"{container_bronze}/jolpica/results_*.json"
    df_results_raw = spark.read.json(path_results)

    print("📜 Schema do Arquivo JSON:")
    df_results_raw.printSchema()

    print("Visualizando as 5 primeiras linhas do JSON cru:")
    display(df_results_raw.limit(5))

except Exception as e:
    print(f"❌ Erro na leitura dos arquivos: {e}")

🔍 Iniciando Inspeção dos Resultados...
📜 Schema do Arquivo JSON:
root
 |-- MRData: struct (nullable = true)
 |    |-- RaceTable: struct (nullable = true)
 |    |    |-- Races: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- Circuit: struct (nullable = true)
 |    |    |    |    |    |-- Location: struct (nullable = true)
 |    |    |    |    |    |    |-- country: string (nullable = true)
 |    |    |    |    |    |    |-- lat: string (nullable = true)
 |    |    |    |    |    |    |-- locality: string (nullable = true)
 |    |    |    |    |    |    |-- long: string (nullable = true)
 |    |    |    |    |    |-- circuitId: string (nullable = true)
 |    |    |    |    |    |-- circuitName: string (nullable = true)
 |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- Results: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    | 

MRData


In [0]:
print("🚀 Iniciando Transformação para Camada Prata...")

try:
    # EXPLODE CORRIDAS
    # "Abre" o array de corridas para que cada corrida vire uma linha
    # O schema mostra que 'Races' está dentro de 'MRData.RaceTable'
    df_races = df_results_raw \
        .select(explode(col("MRData.RaceTable.Races")).alias("race_data"))
        
    # EXPLODE RESULTADOS
    # "Abre" o array de resultados que está DENTRO de cada corrida
    # Mantemos 'race_data' para não perder a info de qual corrida aquele resultado pertence
    df_results_exploded = df_races \
        .select(
            col("race_data"),
            explode(col("race_data.Results")).alias("result_data")
        )
        
    # SELEÇÃO FINAL E TIPAGEM (CAST)
    df_results_silver = df_results_exploded.select(
        # --- DADOS DA CORRIDA (Vêm de race_data) ---
        col("race_data.season").cast("int").alias("season"),
        col("race_data.round").cast("int").alias("round"),
        col("race_data.raceName").alias("race_name"),
        col("race_data.date").cast("date").alias("race_date"),
        
        # Dados do Circuito (Note o caminho: Circuit -> circuitId)
        col("race_data.Circuit.circuitId").alias("circuit_id"),
        col("race_data.Circuit.circuitName").alias("circuit_name"),
        # O Schema mostra que country está dentro de Location
        col("race_data.Circuit.Location.country").alias("country"), 
        
        # --- DADOS DO RESULTADO (Vêm de result_data) ---
        col("result_data.position").cast("int").alias("position"),
        col("result_data.points").cast("float").alias("points"),
        col("result_data.laps").cast("int").alias("laps"),
        col("result_data.status").alias("status"),
        col("result_data.FastestLap.Time.time").alias("fastest_lap_time"),
        col("result_data.FastestLap.rank").cast("int").alias("fastest_lap_rank"),
        
        # --- CHAVES ESTRANGEIRAS (IDs) ---
        col("result_data.Driver.driverId").alias("driver_id"), 
        col("result_data.Constructor.constructorId").alias("team_id"), 
        current_timestamp().alias("ingestion_date")
    )

    df_results_silver = df_results_silver.dropDuplicates(['season', 'round', 'driver_id'])
    caminho_silver = f"{container_silver}/results"
    
    df_results_silver.write.format("delta").mode("overwrite").save(caminho_silver)
    
    print(f"✅ Sucesso! Tabela salva em: {caminho_silver}")
    print("Preview Final:")
    display(df_results_silver.limit(5))

except Exception as e:
    print(f"❌ Erro durante a transformação: {e}")

🚀 Iniciando Transformação para Camada Prata...
✅ Sucesso! Tabela salva em: abfss://silver@f1datalakecarol2026.dfs.core.windows.net/results
Preview Final:


season,round,race_name,race_date,circuit_id,circuit_name,country,position,points,laps,status,fastest_lap_time,fastest_lap_rank,driver_id,team_id,ingestion_date
2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,15,0.0,56,Lapped,1:35.723,17,albon,williams,2026-01-03T17:58:51.875813Z
2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,9,2.0,57,Finished,1:34.199,3,alonso,aston_martin,2026-01-03T17:58:51.875813Z
2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,19,0.0,56,Lapped,1:36.202,19,bottas,sauber,2026-01-03T17:58:51.875813Z
2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,18,0.0,56,Lapped,1:34.805,9,gasly,alpine,2026-01-03T17:58:51.875813Z
2024,1,Bahrain Grand Prix,2024-03-02,bahrain,Bahrain International Circuit,Bahrain,7,6.0,57,Finished,1:34.722,7,hamilton,mercedes,2026-01-03T17:58:51.875813Z


# ✅ Conclusão da Camada Silver

O processamento da camada Prata foi finalizado com sucesso. Os dados agora estão estruturados, tipados e salvos em formato **Delta Lake**.

**Resumo dos Entregáveis:**
1.  **`drivers`**: Tabela Dimensão deduplicada com cadastro de pilotos.
2.  **`constructors`**: Tabela Dimensão deduplicada com cadastro de equipes.
3.  **`results`**: Tabela Fato contendo métricas de corrida, particionada por `season` para alta performance.

**Próximos Passos:**
No próximo notebook (**Camada Gold**), aplicaremos a **Modelagem Dimensional (Star Schema)** para deixar esses dados prontos para análise de negócios e Power BI.

In [0]:
# Validação Final: Listando os arquivos gerados no Data Lake
print("📂 Verificando arquivos na Camada Silver:")

try:
    # Lista as pastas criadas
    files = dbutils.fs.ls(container_silver)
    
    for f in files:
        print(f"✅ Tabela encontrada: {f.name} | Caminho: {f.path}")
        
except Exception as e:
    print(f"Erro na validação: {e}")

📂 Verificando arquivos na Camada Silver:
✅ Tabela encontrada: constructors/ | Caminho: abfss://silver@f1datalakecarol2026.dfs.core.windows.net/constructors/
✅ Tabela encontrada: drivers/ | Caminho: abfss://silver@f1datalakecarol2026.dfs.core.windows.net/drivers/
✅ Tabela encontrada: results/ | Caminho: abfss://silver@f1datalakecarol2026.dfs.core.windows.net/results/
